In [102]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Data Preprocessing

In [103]:
# 1. LOAD THE DATA
# Note: You accidentally loaded Data_Klaim twice in your prompt. I fixed the second one.
claims_df = pd.read_csv('data/Data_Klaim.csv')
polis_df = pd.read_csv('data/Data_Polis.csv')

In [104]:
claims_df.head()

,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,Nominal Klaim Yang Disetujui,Nominal Biaya RS Yang Terjadi,Lokasi RS
0,C-0001-M,POL-0176,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,2024-07-08,2024-05-27,2024-05-27,28093653.000000,6143947.680000,Singapore
1,C-0002-M,POL-3288,R,OP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2024-08-06,2024-07-15,2024-07-15,80987278.000000,82309522.450000,Malaysia
2,C-0003-M,POL-1786,R,OP,C18.9,"MALIGNANT NEOPLASM, COLON, UNSPECIFIED",PAID,2024-10-17,2024-05-16,2024-05-16,183047130.000000,192859905.000000,Singapore
3,C-0004-M,POL-1786,R,OP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2024-09-03,2024-07-18,2024-07-18,191424386.000000,191424385.670000,Singapore
4,C-0005-M,POL-2778,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,NaN,2024-06-06,2024-06-06,138936357.000000,138936357.010000,Singapore


In [105]:
polis_df.head()

,Nomor Polis,Plan Code,Gender,Tanggal Lahir,Tanggal Efektif Polis,Domisili
0,POL-0001,M-003,M,19640811,20140603,JAKARTA
1,POL-0002,M-003,M,19710730,20140603,JAKARTA
2,POL-0003,M-001,M,19790821,20160808,JAKARTA
3,POL-0004,M-003,M,20140724,20160811,JAKARTA
4,POL-0005,M-001,F,19810114,20150828,JAKARTA


In [106]:
# 2. CALCULATE EXPOSURE FROM POLIS TABLE (CRITICAL FOR FREQUENCY)
# Convert Tanggal Efektif Polis (format YYYYMMDD) to datetime
polis_df['Tanggal Efektif Polis'] = pd.to_datetime(polis_df['Tanggal Efektif Polis'], format='%Y%m%d', errors='coerce')
polis_df.head()

,Nomor Polis,Plan Code,Gender,Tanggal Lahir,Tanggal Efektif Polis,Domisili
0,POL-0001,M-003,M,19640811,2014-06-03,JAKARTA
1,POL-0002,M-003,M,19710730,2014-06-03,JAKARTA
2,POL-0003,M-001,M,19790821,2016-08-08,JAKARTA
3,POL-0004,M-003,M,20140724,2016-08-11,JAKARTA
4,POL-0005,M-001,F,19810114,2015-08-28,JAKARTA


In [107]:
# Get min start date and max claim date to build a full timeline
min_date = polis_df['Tanggal Efektif Polis'].min()
claims_df['Tanggal Pasien Masuk RS'] = pd.to_datetime(claims_df['Tanggal Pasien Masuk RS'], errors='coerce')
max_date = claims_df['Tanggal Pasien Masuk RS'].max()

print(min_date)
print(max_date)

2011-12-05 00:00:00
2025-07-31 00:00:00


In [108]:
# Create a continuous monthly timeline and calculate cumulative active policies
all_months = pd.period_range(start=min_date, end=max_date, freq='M')
exposure_df = pd.DataFrame({'Period': all_months})
exposure_df.head()

,Period
0,2011-12
1,2012-01
2,2012-02
3,2012-03
4,2012-04


In [109]:
# Count how many new policies started each month
polis_df['Period'] = polis_df['Tanggal Efektif Polis'].dt.to_period('M')
monthly_sales = polis_df.groupby('Period').size().reset_index(name='New_Policies')
monthly_sales.head()

,Period,New_Policies
0,2011-12,42
1,2012-01,28
2,2012-02,46
3,2012-03,57
4,2012-04,41


In [110]:
# Merge timeline with sales and calculate cumulative exposure
exposure_df = pd.merge(exposure_df, monthly_sales, on='Period', how='left')
exposure_df['New_Policies'] = exposure_df['New_Policies'].fillna(0)
exposure_df['Exposure'] = exposure_df['New_Policies'].cumsum() # Total Active Policies
exposure_df.head()

,Period,New_Policies,Exposure
0,2011-12,42.000000,42.000000
1,2012-01,28.000000,70.000000
2,2012-02,46.000000,116.000000
3,2012-03,57.000000,173.000000
4,2012-04,41.000000,214.000000


In [111]:
# 3. PREPARE & SORT CLAIMS DATA
# Sort by Tanggal Pasien Masuk RS as requested
claims_df = claims_df.sort_values('Tanggal Pasien Masuk RS')
claims_df.head()

,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,Nominal Klaim Yang Disetujui,Nominal Biaya RS Yang Terjadi,Lokasi RS
2810,C-3627-M,POL-3872,C,IP,B34.2,"CORONAVIRUS INFECTION, UNSPECIFIED SITE",PAID,2024-04-25,2024-01-01,2024-01-07,14741310.000000,15025749.000000,Indonesia
3006,C-3836-M,POL-2078,C,OP,N18.5,"CHRONIC KIDNEY DISEASE, STAGE 5",PAID,2024-04-05,2024-01-02,2024-01-02,2769687.000000,3400335.000000,Indonesia
2816,C-3636-M,POL-3932,C,IP,H82,VERTIGINOUS SYNDROMES IN DISEASES CLASSIFIED E...,PAID,2024-02-06,2024-01-02,2024-01-07,36417500.000000,36417500.000000,Indonesia
1948,C-2519-M,POL-2436,R,IP,H35,OTHER RETINAL DISORDERS,PAID,2024-01-17,2024-01-02,2024-01-02,15667000.000000,15667000.000000,Indonesia
144,C-0245-M,POL-2200,C,OP,N23,UNSPECIFIED RENAL COLIC,PAID,2024-04-25,2024-01-02,2024-01-02,1839000.000000,1870000.000000,Indonesia


In [112]:
# Extract Month, Year, and Period for grouping
claims_df['Year'] = claims_df['Tanggal Pasien Masuk RS'].dt.year
claims_df['Month'] = claims_df['Tanggal Pasien Masuk RS'].dt.month
claims_df['Period'] = claims_df['Tanggal Pasien Masuk RS'].dt.to_period('M')
claims_df.head()

,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,Nominal Klaim Yang Disetujui,Nominal Biaya RS Yang Terjadi,Lokasi RS,Year,Month,Period
2810,C-3627-M,POL-3872,C,IP,B34.2,"CORONAVIRUS INFECTION, UNSPECIFIED SITE",PAID,2024-04-25,2024-01-01,2024-01-07,14741310.000000,15025749.000000,Indonesia,2024,1,2024-01
3006,C-3836-M,POL-2078,C,OP,N18.5,"CHRONIC KIDNEY DISEASE, STAGE 5",PAID,2024-04-05,2024-01-02,2024-01-02,2769687.000000,3400335.000000,Indonesia,2024,1,2024-01
2816,C-3636-M,POL-3932,C,IP,H82,VERTIGINOUS SYNDROMES IN DISEASES CLASSIFIED E...,PAID,2024-02-06,2024-01-02,2024-01-07,36417500.000000,36417500.000000,Indonesia,2024,1,2024-01
1948,C-2519-M,POL-2436,R,IP,H35,OTHER RETINAL DISORDERS,PAID,2024-01-17,2024-01-02,2024-01-02,15667000.000000,15667000.000000,Indonesia,2024,1,2024-01
144,C-0245-M,POL-2200,C,OP,N23,UNSPECIFIED RENAL COLIC,PAID,2024-04-25,2024-01-02,2024-01-02,1839000.000000,1870000.000000,Indonesia,2024,1,2024-01


In [113]:
# Identify Inpatient (IP) vs Outpatient (OP, ODC, ODS)
claims_df['Is_IP'] = claims_df['Inpatient/Outpatient'].isin(['IP']).astype(int)
claims_df['Is_OP'] = claims_df['Inpatient/Outpatient'].isin(['OP', 'ODC', 'ODS']).astype(int)
claims_df.head()

,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,Nominal Klaim Yang Disetujui,Nominal Biaya RS Yang Terjadi,Lokasi RS,Year,Month,Period,Is_IP,Is_OP
2810,C-3627-M,POL-3872,C,IP,B34.2,"CORONAVIRUS INFECTION, UNSPECIFIED SITE",PAID,2024-04-25,2024-01-01,2024-01-07,14741310.000000,15025749.000000,Indonesia,2024,1,2024-01,1,0
3006,C-3836-M,POL-2078,C,OP,N18.5,"CHRONIC KIDNEY DISEASE, STAGE 5",PAID,2024-04-05,2024-01-02,2024-01-02,2769687.000000,3400335.000000,Indonesia,2024,1,2024-01,0,1
2816,C-3636-M,POL-3932,C,IP,H82,VERTIGINOUS SYNDROMES IN DISEASES CLASSIFIED E...,PAID,2024-02-06,2024-01-02,2024-01-07,36417500.000000,36417500.000000,Indonesia,2024,1,2024-01,1,0
1948,C-2519-M,POL-2436,R,IP,H35,OTHER RETINAL DISORDERS,PAID,2024-01-17,2024-01-02,2024-01-02,15667000.000000,15667000.000000,Indonesia,2024,1,2024-01,1,0
144,C-0245-M,POL-2200,C,OP,N23,UNSPECIFIED RENAL COLIC,PAID,2024-04-25,2024-01-02,2024-01-02,1839000.000000,1870000.000000,Indonesia,2024,1,2024-01,0,1


In [114]:
# 4. AGGREGATE PER MONTH
agg_df = claims_df.groupby(['Period', 'Year', 'Month']).agg(
    Jumlah_Inpatient=('Is_IP', 'sum'),
    Jumlah_Outpatient=('Is_OP', 'sum'),
    Jumlah_Klaim=('Claim ID', 'count'),
    Total_Nominal_Klaim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
agg_df.head()

,Period,Year,Month,Jumlah_Inpatient,Jumlah_Outpatient,Jumlah_Klaim,Total_Nominal_Klaim
0,2024-01,2024,1,213,81,302,20260981490.193146
1,2024-02,2024,2,140,60,208,13859645460.103493
2,2024-03,2024,3,196,82,278,14311258092.349430
3,2024-04,2024,4,160,79,239,11441062778.549999
4,2024-05,2024,5,163,100,263,12211461820.570000


In [115]:
# 5. MERGE CLAIMS WITH EXPOSURE & CALCULATE METRICS
final_df = pd.merge(agg_df, exposure_df[['Period', 'Exposure']], on='Period', how='left')
final_df.head()

,Period,Year,Month,Jumlah_Inpatient,Jumlah_Outpatient,Jumlah_Klaim,Total_Nominal_Klaim,Exposure
0,2024-01,2024,1,213,81,302,20260981490.193146,4096.000000
1,2024-02,2024,2,140,60,208,13859645460.103493,4096.000000
2,2024-03,2024,3,196,82,278,14311258092.349430,4096.000000
3,2024-04,2024,4,160,79,239,11441062778.549999,4096.000000
4,2024-05,2024,5,163,100,263,12211461820.570000,4096.000000


In [116]:
# Metric 1: Ratio Inpatient / Outpatient (using np.where to avoid divide-by-zero errors)
final_df['Ratio Inpatient / Outpatient'] = np.where(
    final_df['Jumlah_Outpatient'] == 0, 
    np.nan, 
    final_df['Jumlah_Inpatient'] / final_df['Jumlah_Outpatient']
)
final_df.head()

,Period,Year,Month,Jumlah_Inpatient,Jumlah_Outpatient,Jumlah_Klaim,Total_Nominal_Klaim,Exposure,Ratio Inpatient / Outpatient
0,2024-01,2024,1,213,81,302,20260981490.193146,4096.000000,2.629630
1,2024-02,2024,2,140,60,208,13859645460.103493,4096.000000,2.333333
2,2024-03,2024,3,196,82,278,14311258092.349430,4096.000000,2.390244
3,2024-04,2024,4,160,79,239,11441062778.549999,4096.000000,2.025316
4,2024-05,2024,5,163,100,263,12211461820.570000,4096.000000,1.630000


In [117]:
# Metric 2: True Frequency Rate = (Jumlah Klaim / Active Policies)
final_df['Freq_Rate'] = final_df['Jumlah_Klaim'] / final_df['Exposure']
final_df.head()

,Period,Year,Month,Jumlah_Inpatient,Jumlah_Outpatient,Jumlah_Klaim,Total_Nominal_Klaim,Exposure,Ratio Inpatient / Outpatient,Freq_Rate
0,2024-01,2024,1,213,81,302,20260981490.193146,4096.000000,2.629630,0.073730
1,2024-02,2024,2,140,60,208,13859645460.103493,4096.000000,2.333333,0.050781
2,2024-03,2024,3,196,82,278,14311258092.349430,4096.000000,2.390244,0.067871
3,2024-04,2024,4,160,79,239,11441062778.549999,4096.000000,2.025316,0.058350
4,2024-05,2024,5,163,100,263,12211461820.570000,4096.000000,1.630000,0.064209


In [118]:
# Metric 3: Severity = (Total Nominal / Jumlah Klaim)
final_df['Severity'] = final_df['Total_Nominal_Klaim'] / final_df['Jumlah_Klaim']
final_df.head()

,Period,Year,Month,Jumlah_Inpatient,Jumlah_Outpatient,Jumlah_Klaim,Total_Nominal_Klaim,Exposure,Ratio Inpatient / Outpatient,Freq_Rate,Severity
0,2024-01,2024,1,213,81,302,20260981490.193146,4096.000000,2.629630,0.073730,67089342.682759
1,2024-02,2024,2,140,60,208,13859645460.103493,4096.000000,2.333333,0.050781,66632910.865882
2,2024-03,2024,3,196,82,278,14311258092.349430,4096.000000,2.390244,0.067871,51479345.655933
3,2024-04,2024,4,160,79,239,11441062778.549999,4096.000000,2.025316,0.058350,47870555.558787
4,2024-05,2024,5,163,100,263,12211461820.570000,4096.000000,1.630000,0.064209,46431413.766426


In [119]:
# 6. CLEAN UP AND RENAME COLUMNS
final_df = final_df[[
    'Period','Month', 'Year', 'Jumlah_Inpatient', 'Jumlah_Outpatient', 
    'Ratio Inpatient / Outpatient', 'Exposure', 'Jumlah_Klaim', 'Freq_Rate', 'Severity', 'Total_Nominal_Klaim'
]].copy()
final_df.head()

,Period,Month,Year,Jumlah_Inpatient,Jumlah_Outpatient,Ratio Inpatient / Outpatient,Exposure,Jumlah_Klaim,Freq_Rate,Severity,Total_Nominal_Klaim
0,2024-01,1,2024,213,81,2.629630,4096.000000,302,0.073730,67089342.682759,20260981490.193146
1,2024-02,2,2024,140,60,2.333333,4096.000000,208,0.050781,66632910.865882,13859645460.103493
2,2024-03,3,2024,196,82,2.390244,4096.000000,278,0.067871,51479345.655933,14311258092.349430
3,2024-04,4,2024,160,79,2.025316,4096.000000,239,0.058350,47870555.558787,11441062778.549999
4,2024-05,5,2024,163,100,1.630000,4096.000000,263,0.064209,46431413.766426,12211461820.570000


In [120]:
final_df = final_df.rename(columns={
    'Jumlah_Inpatient': 'Jumlah Inpatient',
    'Jumlah_Outpatient': 'Jumlah Outpatient',
    'Jumlah_Klaim': 'Frequency',
    'Total_Nominal_Klaim': 'Total Nominal Klaim'
})
final_df.head()

,Period,Month,Year,Jumlah Inpatient,Jumlah Outpatient,Ratio Inpatient / Outpatient,Exposure,Frequency,Freq_Rate,Severity,Total Nominal Klaim
0,2024-01,1,2024,213,81,2.629630,4096.000000,302,0.073730,67089342.682759,20260981490.193146
1,2024-02,2,2024,140,60,2.333333,4096.000000,208,0.050781,66632910.865882,13859645460.103493
2,2024-03,3,2024,196,82,2.390244,4096.000000,278,0.067871,51479345.655933,14311258092.349430
3,2024-04,4,2024,160,79,2.025316,4096.000000,239,0.058350,47870555.558787,11441062778.549999
4,2024-05,5,2024,163,100,1.630000,4096.000000,263,0.064209,46431413.766426,12211461820.570000


In [121]:

# Add Final "Total Klaim" column as requested
final_df['Total Klaim'] = final_df['Total Nominal Klaim']

# Remove any empty rows where date couldn't be parsed
final_df = final_df.dropna(subset=['Year', 'Month']).copy()
final_df['Year'] = final_df['Year'].astype(int)
final_df['Month'] = final_df['Month'].astype(int)
final_df.head()

,Period,Month,Year,Jumlah Inpatient,Jumlah Outpatient,Ratio Inpatient / Outpatient,Exposure,Frequency,Freq_Rate,Severity,Total Nominal Klaim,Total Klaim
0,2024-01,1,2024,213,81,2.629630,4096.000000,302,0.073730,67089342.682759,20260981490.193146,20260981490.193146
1,2024-02,2,2024,140,60,2.333333,4096.000000,208,0.050781,66632910.865882,13859645460.103493,13859645460.103493
2,2024-03,3,2024,196,82,2.390244,4096.000000,278,0.067871,51479345.655933,14311258092.349430,14311258092.349430
3,2024-04,4,2024,160,79,2.025316,4096.000000,239,0.058350,47870555.558787,11441062778.549999,11441062778.549999
4,2024-05,5,2024,163,100,1.630000,4096.000000,263,0.064209,46431413.766426,12211461820.570000,12211461820.570000


In [122]:
final_df

,Period,Month,Year,Jumlah Inpatient,Jumlah Outpatient,Ratio Inpatient / Outpatient,Exposure,Frequency,Freq_Rate,Severity,Total Nominal Klaim,Total Klaim
0,2024-01,1,2024,213,81,2.629630,4096.000000,302,0.073730,67089342.682759,20260981490.193146,20260981490.193146
1,2024-02,2,2024,140,60,2.333333,4096.000000,208,0.050781,66632910.865882,13859645460.103493,13859645460.103493
2,2024-03,3,2024,196,82,2.390244,4096.000000,278,0.067871,51479345.655933,14311258092.349430,14311258092.349430
3,2024-04,4,2024,160,79,2.025316,4096.000000,239,0.058350,47870555.558787,11441062778.549999,11441062778.549999
4,2024-05,5,2024,163,100,1.630000,4096.000000,263,0.064209,46431413.766426,12211461820.570000,12211461820.570000
5,2024-06,6,2024,106,119,0.890756,4096.000000,225,0.054932,53889631.911244,12125167180.030001,12125167180.030001
6,2024-07,7,2024,112,144,0.777778,4096.000000,257,0.062744,58251037.858949,14970516729.750000,14970516729.750000
7,2024-08,8,2024,95,130,0.730769,4096.000000,228,0.055664,59267262.543465,13512935859.910000,13512935859.910000
8,2024-09,9,2024,89,117,0.760684,4096.000000,208,0.050781,58962110.902019,12264119067.620001,12264119067.620001
9,2024-10,10,2024,107,164,0.652439,4096.000000,274,0.066895,46281627.239781,12681165863.700001,12681165863.700001


Model A Training Without Inpatient/Outpatient Feature

In [123]:
from sklearn.ensemble import GradientBoostingRegressor

In [124]:
def create_lag_features(data, target_col):
    df_feat = data.copy()
    # Lag Features (1, 2, 3 months back)
    for lag in [1, 2, 3]:
        df_feat[f'lag_{lag}'] = df_feat[target_col].shift(lag)
    # Rolling Feature (Mean of last 3 months)
    df_feat['rolling_mean_3'] = df_feat[target_col].shift(1).rolling(window=3).mean()
    return df_feat.dropna()

In [125]:
# 1. Train Model A - Frequency
train_freq = create_lag_features(final_df, 'Frequency')
X_freq = train_freq[['lag_1', 'lag_2', 'lag_3', 'rolling_mean_3']]
y_freq = train_freq['Frequency']

In [126]:
modelA_freq = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
modelA_freq.fit(X_freq, y_freq)

,loss,'squared_error'
,learning_rate,0.05
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [127]:
# 2. Train Model A - Severity
train_sev = create_lag_features(final_df, 'Severity')
X_sev = train_sev[['lag_1', 'lag_2', 'lag_3', 'rolling_mean_3']]
y_sev = train_sev['Severity']

In [128]:
modelA_sev = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
modelA_sev.fit(X_sev, y_sev)

,loss,'squared_error'
,learning_rate,0.05
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [129]:
# --- STEP 3: RECURSIVE FORECASTING (Aug - Dec 2025) ---
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
forecast_range = pd.period_range(start=final_df['Period'].max() + 1, end=target_end, freq='M')

In [130]:
# History DataFrame (starts with actual data, grows with predictions)
history_df = final_df.copy()
current_exposure = history_df['Exposure'].iloc[-1] # Assuming constant exposure for forecast

predictions = []

In [131]:
for month in forecast_range:
    # 1. Build Features from History (Last 3 rows)
    last_3 = history_df.tail(3)
    
    # Feature Vector for Frequency
    freq_feats = np.array([[
        last_3['Frequency'].iloc[-1], # lag_1
        last_3['Frequency'].iloc[-2], # lag_2
        last_3['Frequency'].iloc[-3], # lag_3
        last_3['Frequency'].mean()    # rolling_mean_3
    ]])
    
    # Feature Vector for Severity
    sev_feats = np.array([[
        last_3['Severity'].iloc[-1],
        last_3['Severity'].iloc[-2],
        last_3['Severity'].iloc[-3],
        last_3['Severity'].mean()
    ]])
    
    # 2. Predict
    pred_frequency = modelA_freq.predict(freq_feats)[0]
    pred_sev = modelA_sev.predict(sev_feats)[0]
    
    # Safety: Ensure no negative predictions
    pred_frequency = max(0, pred_frequency)
    pred_sev = max(0, pred_sev)
    
    # 3. Calculate Derived Metrics
    pred_claim_count = pred_frequency
    pred_total_claim = pred_claim_count * pred_sev
    
    # 4. Append to History
    new_row = pd.DataFrame([{
        'Month_Period': month,
        'Claim_Count': pred_claim_count,
        'Total_Nominal': pred_total_claim,
        'Exposure': current_exposure,
        'Frequency': pred_frequency,
        'Severity': pred_sev
    }])
    history_df = pd.concat([history_df, new_row], ignore_index=True)
    
    # 5. Store if in Target Range (Aug - Dec)
    if month >= target_start:
        predictions.append({
            'Month': str(month),
            'Jumlah Klaim': int(round(pred_claim_count)),
            'Total Klaim': pred_total_claim,
            'Frequency': pred_frequency,
            'Severity': pred_sev
        })

c:\Users\hi\anaconda3\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\hi\anaconda3\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\hi\anaconda3\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\hi\anaconda3\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\hi\anaconda3\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have val

In [132]:
# --- STEP 4: FINAL RESULT ---
modelA_result_df = pd.DataFrame(predictions)

# Formatting for Display
pd.options.display.float_format = '{:f}'.format
print("Model A Prediction Results (Aug - Dec 2025):")
print(modelA_result_df)

Model A Prediction Results (Aug - Dec 2025):
     Month  Jumlah Klaim        Total Klaim  Frequency        Severity
0  2025-08           224 13006524431.265368 224.166173 58021798.163976
1  2025-09           252 12724009375.474773 252.393986 50413282.761677
2  2025-10           228 12129944638.245440 228.127103 53171869.905693
3  2025-11           209 11091678351.623657 208.801263 53120743.572480
4  2025-12           272 15872474142.575233 272.252172 58300633.602397


In [133]:
# Create a list to store the formatted rows
formatted_data = []

for index, row in modelA_result_df.iterrows():
    # Format the month from '2025-08' to '2025_08'
    month_str = str(row['Month']).replace('-', '_')
    
    # Append Frequency
    formatted_data.append({
        'id': f"{month_str}_Claim_Frequency",
        'value': row['Frequency']
    })
    
    # Append Severity
    formatted_data.append({
        'id': f"{month_str}_Claim_Severity",
        'value': row['Severity']
    })
    
    # Append Total Claim
    formatted_data.append({
        'id': f"{month_str}_Total_Claim",
        'value': row['Total Klaim']
    })

# Create the final dataframe
submission_df = pd.DataFrame(formatted_data)

# Display
print(submission_df)

# Export to CSV without the index number
submission_df.to_csv('submission_gb.csv', index=False)

print("File 'submission_gb.csv' saved successfully.")

                         id              value
0   2025_08_Claim_Frequency         224.166173
1    2025_08_Claim_Severity    58021798.163976
2       2025_08_Total_Claim 13006524431.265368
3   2025_09_Claim_Frequency         252.393986
4    2025_09_Claim_Severity    50413282.761677
5       2025_09_Total_Claim 12724009375.474773
6   2025_10_Claim_Frequency         228.127103
7    2025_10_Claim_Severity    53171869.905693
8       2025_10_Total_Claim 12129944638.245440
9   2025_11_Claim_Frequency         208.801263
10   2025_11_Claim_Severity    53120743.572480
11      2025_11_Total_Claim 11091678351.623657
12  2025_12_Claim_Frequency         272.252172
13   2025_12_Claim_Severity    58300633.602397
14      2025_12_Total_Claim 15872474142.575233
File 'submission_gb.csv' saved successfully.
